# ELT Case: Snowflake and Python

Purchase orders say what goods should cost. Supplier invoices say what was actually billed.
This notebook builds a pipeline that puts those two numbers side by side, from five sources in
five different formats, and then answers eight questions about what it shows.

Everything is driven from Python, but the work happens **inside Snowflake** - the code is almost
entirely `cs.execute(...)`.

**How this works.** Fill this notebook in, then commit and push it. The pushed notebook is the
submission - there is nothing else to hand in.

Part 1 builds the pipeline, one step per section. Part 2 answers eight questions about what it
shows.

Comments in the code are enough for Part 1. Each Part 2 question has an empty markdown cell under
it for your own reading of the result - **that is optional**, but some of these numbers have a
story behind them and a sentence saying what you make of one is worth more than the number alone.

Two practical things. The notebook should run **top to bottom on a clean kernel** - if a cell only
works because of something you ran earlier and deleted, it will not work for whoever opens it
next. And **do not commit your Snowflake password**; a secret in a commit stays in the git history
even after you delete the line.


## Part 1 - Build the pipeline

### Step 1 - Connect to Snowflake

Do **not** hard-code your password. This notebook is going into version control, and a secret in
a commit stays in the history even after you delete the line. Set the three environment variables
before launching VS Code, or read them from a file you keep out of the repository.


In [1]:
import os
import glob
import re
import csv
import pathlib
import snowflake.connector
from dotenv import load_dotenv

load_dotenv(override=True)

# connect to Snowflake and create a cursor

conn = snowflake.connector.connect(
    user=os.getenv("USERNAME"),
    password=os.getenv("PASSWORD"),
    account=os.getenv("ACCOUNT_STRING"),
)

cs = conn.cursor()


### Step 2 - Create the Snowflake objects

A warehouse for compute, a database for the case, and two schemas: `STAGE` for the internal
stages and file formats, `CORE` for the tables and views we build from them. Separating the
landing area from the modelled tables keeps it obvious which objects are raw and which are
derived.


In [2]:
# insert code: create the warehouse, database and the STAGE and CORE schemas
cs.execute("CREATE WAREHOUSE IF NOT EXISTS mgta464_wh")
cs.execute("CREATE DATABASE  IF NOT EXISTS po_case")
cs.execute("USE DATABASE po_case")

cs.execute("CREATE SCHEMA IF NOT EXISTS po_case.STAGE")  # stages, file formats
cs.execute("CREATE SCHEMA IF NOT EXISTS po_case.CORE")  # tables, views

for stmt in ("USE WAREHOUSE mgta464_wh", "USE DATABASE po_case", "USE SCHEMA CORE"):
    cs.execute(stmt)

### Step 3 - Load the 41 purchase order files

The files are at line-item level, one per month. Three things to get right:

- **Skip the header row.** `SKIP_HEADER = 1` in the file format, or 41 header rows become data.
- **Do the transformation in the `COPY INTO`.** Select the columns you want and cast them there,
  rather than loading everything as text and fixing it afterwards.
- **Automate the `PUT`.** Iterate with `glob`; stage into `year/month` folders. It makes no
  practical difference at this size, but it is the right habit for data that arrives over time.

Columns dropped as not useful: `Comments` and `InternalComments` are almost entirely NULL,
`LastEditedBy` / `LastEditedWhen` (and their `Right_` duplicates) are audit fields, and
`PackageTypeID` carries a single value.

One data note: a few rows carry the date `2/29/2022`, which does not exist - 2022 was not a leap
year. `TRY_TO_DATE` returns NULL for those rather than failing the load, which is exactly why it
is used here instead of `TO_DATE`.


In [3]:
# insert code: create the file format, the internal stage, and the CORE.PURCHASES table
cs.execute("""
CREATE OR REPLACE TABLE CORE.PURCHASES (
    purchaseorderid            INTEGER,
    supplierid                 INTEGER,
    orderdate                  DATE,
    deliverymethodid           INTEGER,
    expecteddeliverydate       DATE,
    supplierreference          VARCHAR,
    isorderfinalized           BOOLEAN,
    lasteditedby               INTEGER,
    purchaseorderlineid        INTEGER,
    stockitemid                INTEGER,
    orderedouters              INTEGER,
    description                VARCHAR,
    receivedouters             INTEGER,
    packagetypeid              INTEGER,
    expectedunitpriceperouter  NUMBER(18,2),
    lastreceiptdate            DATE,
    isorderlinefinalized       BOOLEAN,
    right_lasteditedby         INTEGER
)
""")

In [4]:
# insert code: PUT every purchases csv into the stage, partitioned by year and month
paths = sorted(glob.glob("data/Monthly PO Data/*.csv"))
print(len(paths), "files")

for path in paths:
    stem = os.path.basename(path)  # 2019-1.csv
    year, month = re.match(r"(\d{4})-(\d{1,2})\.csv", stem).groups()
    abs_path = os.path.abspath(path)
    cs.execute(
        f"PUT 'file://{abs_path}' "
        f"@STAGE.po_stage/purchases/{year}/{int(month):02d} "
        "AUTO_COMPRESS=TRUE OVERWRITE=TRUE"
    )

cs.execute("LIST @STAGE.po_stage/purchases/")
print(len(cs.fetchall()), "files staged")

41 files
41 files staged


In [5]:
# insert code: COPY INTO CORE.PURCHASES, selecting columns and casting types in the same step
cs.execute("""
COPY INTO CORE.PURCHASES (
    purchaseorderid, supplierid, orderdate, deliverymethodid,
    expecteddeliverydate, supplierreference, isorderfinalized,
    lasteditedby, purchaseorderlineid, stockitemid,
    orderedouters, description, receivedouters, packagetypeid,
    expectedunitpriceperouter, lastreceiptdate, isorderlinefinalized,
    right_lasteditedby
)
FROM (
  SELECT $1::INTEGER,
         $2::INTEGER,
         TRY_TO_DATE($3),
         $4::INTEGER,
         TRY_TO_DATE($6),
         $7::VARCHAR,
         $8::BOOLEAN,
         $11::INTEGER,
         $13::INTEGER,
         $14::INTEGER,
         $15::INTEGER,
         $16::VARCHAR,
         $17::INTEGER,
         $18::INTEGER,
         $19::NUMBER(18,2),
         TRY_TO_DATE($20),
         $21::BOOLEAN,
         $22::INTEGER
  FROM @STAGE.po_stage/purchases/
)
FILE_FORMAT = (FORMAT_NAME = STAGE.csv_po)
PATTERN = '.*[.]csv[.]gz'
ON_ERROR = 'ABORT_STATEMENT'
""")

cs.execute("""
SELECT COUNT(*)          AS n_rows,
       COUNT(orderdate)  AS with_orderdate,
       MIN(orderdate)    AS first_order,
       MAX(orderdate)    AS last_order
FROM CORE.PURCHASES
""")
print(cs.fetchone())

(8367, 8358, datetime.date(2019, 1, 1), datetime.date(2022, 5, 31))


### Step 4 - Purchase order totals

Roll the line items up to one row per order. `POAmount` is the sum of
`ReceivedOuters * ExpectedUnitPricePerOuter` - **received**, not ordered. The gap between those
two is the whole point of the case.

`OrderDate` and `SupplierID` come along for the ride because they are constant within an order
and every downstream step needs them.


In [6]:
# insert code: build CORE.PURCHASE_ORDER_TOTALS with POAmount
cs.execute("""
CREATE OR REPLACE TABLE CORE.PURCHASE_ORDER_TOTALS AS
SELECT purchaseorderid,
       supplierid,
       orderdate,
       SUM(receivedouters * expectedunitpriceperouter) AS POAmount
FROM CORE.PURCHASES
GROUP BY 1, 2, 3
""")

cs.execute("""
SELECT COUNT(*), COUNT(DISTINCT purchaseorderid) FROM CORE.PURCHASE_ORDER_TOTALS
""")
print(cs.fetchone())

(2074, 2074)


### Step 5 - Load and shred the supplier transactions

The XML lands in a `VARIANT` column first, then `LATERAL FLATTEN` turns each `<row>` into a
Snowflake row and `XMLGET` pulls the elements out of it.

Look at what is in the file before deciding what to keep. It is not all invoices: rows with
`TransactionTypeID` 5 are supplier invoices and carry a `PurchaseOrderID`; rows with
`TransactionTypeID` 7 are payments, have no purchase order, and have a zero ex-tax amount. Both
belong in the table - the join in step 6 is what filters to invoices.


In [7]:
# insert code: file format, stage, raw VARIANT table, and the shredded CORE.SUPPLIER_TRANSACTIONS

# 1. XML file format
cs.execute("CREATE OR REPLACE FILE FORMAT STAGE.xml_txn TYPE = XML")

# 2. stage the file (quotes needed - the filename contains spaces)
cs.execute("CREATE OR REPLACE STAGE STAGE.txn_stage")
xml_path = os.path.abspath("data/Supplier Transactions XML.xml")
cs.execute(
    f"PUT 'file://{xml_path}' @STAGE.txn_stage AUTO_COMPRESS=TRUE OVERWRITE=TRUE"
)

# 3. land the whole document in a single VARIANT row
cs.execute("CREATE OR REPLACE TABLE STAGE.TXN_RAW (v VARIANT)")
cs.execute("""
COPY INTO STAGE.TXN_RAW
FROM @STAGE.txn_stage
FILE_FORMAT = (FORMAT_NAME = STAGE.xml_txn)
ON_ERROR = 'ABORT_STATEMENT'
""")

# 4. flatten each <row> and shred the elements into columns
cs.execute("""
CREATE OR REPLACE TABLE CORE.SUPPLIER_TRANSACTIONS AS
SELECT XMLGET(r.value, 'SupplierTransactionID'):"$"::INTEGER    AS suppliertransactionid,
       XMLGET(r.value, 'SupplierID'):"$"::INTEGER               AS supplierid,
       XMLGET(r.value, 'TransactionTypeID'):"$"::INTEGER        AS transactiontypeid,
       TRY_TO_NUMBER(XMLGET(r.value, 'PurchaseOrderID'):"$"::VARCHAR)       AS purchaseorderid,
       XMLGET(r.value, 'PaymentMethodID'):"$"::INTEGER          AS paymentmethodid,
       TRY_TO_NUMBER(XMLGET(r.value, 'SupplierInvoiceNumber'):"$"::VARCHAR) AS supplierinvoicenumber,
       TRY_TO_DATE(XMLGET(r.value, 'TransactionDate'):"$"::VARCHAR)         AS transactiondate,
       XMLGET(r.value, 'AmountExcludingTax'):"$"::NUMBER(18,2)  AS amountexcludingtax,
       XMLGET(r.value, 'TaxAmount'):"$"::NUMBER(18,2)           AS taxamount,
       XMLGET(r.value, 'TransactionAmount'):"$"::NUMBER(18,2)   AS transactionamount,
       XMLGET(r.value, 'OutstandingBalance'):"$"::NUMBER(18,2)  AS outstandingbalance,
       TRY_TO_DATE(XMLGET(r.value, 'FinalizationDate'):"$"::VARCHAR)        AS finalizationdate,
       XMLGET(r.value, 'IsFinalized'):"$"::VARCHAR::BOOLEAN     AS isfinalized
FROM STAGE.TXN_RAW t,
     LATERAL FLATTEN(input => t.v:"$") r
""")

# checks
cs.execute("SELECT COUNT(*) FROM STAGE.TXN_RAW")
print("raw rows (expect 1):", cs.fetchone())

cs.execute("SELECT COUNT(*) FROM CORE.SUPPLIER_TRANSACTIONS")
print("transactions (expect 2438):", cs.fetchone())

cs.execute("""
SELECT transactiontypeid,
       COUNT(*)               AS n,
       COUNT(purchaseorderid) AS with_po
FROM CORE.SUPPLIER_TRANSACTIONS
GROUP BY 1 ORDER BY 1
""")
for row in cs.fetchall():
    print(row)


raw rows (expect 1): (1,)
transactions (expect 2438): (2438,)
(5, 2072, 2072)
(7, 366, 0)


### Step 6 - Join orders to invoices

Inner join, so only orders that were invoiced survive. `invoiced_vs_quoted` is
`AmountExcludingTax - POAmount`: positive means the supplier billed more than the value of what
arrived.

Watch the grain. The join is order to invoice, one row each - if you join to `CORE.PURCHASES`
instead of the totals table you get one row per **line item** and every difference is counted
several times over.

The case asks for a materialized view. Snowflake materialized views cannot contain joins, so a
table is the right substitute here, exactly as the instructions allow.


In [8]:
# insert code: create purchase_orders_and_invoices with the invoiced_vs_quoted field

cs.execute("""
CREATE OR REPLACE TABLE CORE.PURCHASE_ORDERS_AND_INVOICES AS
SELECT t.purchaseorderid,
       t.supplierid,
       t.transactiondate,
       t.amountexcludingtax,
       p.POAmount,
       t.amountexcludingtax - p.POAmount AS invoiced_vs_quoted
FROM CORE.SUPPLIER_TRANSACTIONS t
JOIN CORE.PURCHASE_ORDER_TOTALS p USING (purchaseorderid)
""")

cs.execute("SELECT COUNT(*) FROM CORE.PURCHASE_ORDERS_AND_INVOICES")
print("expect 2072:", cs.fetchone())

expect 2072: (2072,)


### Step 7 - Bring the supplier data across from Postgres

The data must not pass through Python. Postgres writes it to a file with `COPY ... TO STDOUT`,
and Snowflake picks the file up from a stage.

The `CREATE TABLE` is generated from the file itself rather than typed by hand - a function that
reads the header and samples the data to pick a type per column. That is reusable; a hand-written
`CREATE TABLE` is not.


In [9]:
from datetime import datetime

TEXT_COLUMNS = {"deliverypostalcode", "postalpostalcode"}


def snowflake_type(values):
    """Pick a Snowflake data type for a column, given a sample of its values."""
    vals = [
        v.strip() for v in values if v is not None and v.strip() not in ("", "NULL")
    ]
    if not vals:
        return "VARCHAR"

    def all_parse(fn):
        for v in vals:
            try:
                fn(v)
            except (ValueError, TypeError):
                return False
        return True

    if all_parse(int):
        return "INTEGER"
    if all_parse(float):
        return "NUMBER(18,2)"
    if all_parse(lambda v: datetime.strptime(v, "%Y-%m-%d")):
        return "DATE"
    return "VARCHAR"


def create_table_sql(csv_path, table_name, sample_rows=200):
    """Read a csv header and sample its rows, and return a CREATE TABLE statement."""
    """Read a csv header and sample its rows, and return a CREATE TABLE statement."""
    with open(csv_path, newline="") as f:
        reader = csv.DictReader(f)
        columns = reader.fieldnames
        sample = {c: [] for c in columns}
        for i, row in enumerate(reader):
            if i >= sample_rows:
                break
            for c in columns:
                sample[c].append(row[c])

    defs = []
    for c in columns:
        dtype = "VARCHAR" if c.lower() in TEXT_COLUMNS else snowflake_type(sample[c])
        defs.append(f"    {c} {dtype}")

    return f"CREATE OR REPLACE TABLE {table_name} (\n" + ",\n".join(defs) + "\n)"


In [10]:
# insert code: export supplier_case from postgres to a file, stage it, and load it
import psycopg2

# 1. Postgres writes the file itself - the rows never pass through Python.
#    PGHOST / PGPORT / PGUSER / PGDATABASE are already exported by direnv,
#    so connect() needs no arguments.
out_path = os.path.abspath("data/supplier_case.csv")
with psycopg2.connect() as pg:
    with pg.cursor() as pcur, open(out_path, "w", newline="") as f:
        pcur.copy_expert("COPY supplier_case TO STDOUT WITH CSV HEADER", f)

# 2. generate the CREATE TABLE from the exported file itself
ddl = create_table_sql(out_path, "CORE.SUPPLIER")
print(ddl)
cs.execute(ddl)

# 3. stage the file and load it
cs.execute("""
CREATE OR REPLACE FILE FORMAT STAGE.csv_pg
  TYPE = CSV
  SKIP_HEADER = 1
  FIELD_OPTIONALLY_ENCLOSED_BY = '"'
  EMPTY_FIELD_AS_NULL = TRUE
""")
cs.execute("CREATE OR REPLACE STAGE STAGE.pg_stage")
cs.execute(f"PUT 'file://{out_path}' @STAGE.pg_stage AUTO_COMPRESS=TRUE OVERWRITE=TRUE")
cs.execute("""
COPY INTO CORE.SUPPLIER
FROM @STAGE.pg_stage
FILE_FORMAT = (FORMAT_NAME = STAGE.csv_pg)
ON_ERROR = 'ABORT_STATEMENT'
""")

# checks
cs.execute("SELECT COUNT(*) FROM CORE.SUPPLIER")
print("suppliers (expect 13):", cs.fetchone())

cs.execute("""
SELECT supplierid, suppliername, deliverypostalcode, postalpostalcode
FROM CORE.SUPPLIER ORDER BY supplierid
""")
for row in cs.fetchall():
    print(row)

CREATE OR REPLACE TABLE CORE.SUPPLIER (
    supplierid INTEGER,
    suppliername VARCHAR,
    suppliercategoryid INTEGER,
    primarycontactpersonid INTEGER,
    alternatecontactpersonid INTEGER,
    deliverymethodid INTEGER,
    postalcityid INTEGER,
    supplierreference VARCHAR,
    bankaccountname VARCHAR,
    bankaccountbranch VARCHAR,
    bankaccountcode INTEGER,
    bankaccountnumber INTEGER,
    bankinternationalcode INTEGER,
    paymentdays INTEGER,
    internalcomments VARCHAR,
    phonenumber VARCHAR,
    faxnumber VARCHAR,
    websiteurl VARCHAR,
    deliveryaddressline1 VARCHAR,
    deliveryaddressline2 VARCHAR,
    deliverypostalcode VARCHAR,
    deliverylocation VARCHAR,
    postaladdressline1 VARCHAR,
    postaladdressline2 VARCHAR,
    postalpostalcode VARCHAR,
    lasteditedby INTEGER,
    validfrom VARCHAR,
    validto VARCHAR
)
suppliers (expect 13): (13,)
(1, 'A Datum Corporation', '22202', '22202')
(2, 'Contoso, Ltd.', '80125', '80125')
(3, 'Consolidated Messenger

### Step 8 - Weather

The Marketplace subscription is the one thing that cannot be driven from Python - do it once in
the Snowflake web interface (**Data Products → Marketplace → NOAA → Weather & Environment →
Get**). After that, everything is SQL again.

Then three pieces of work:

1. **Load the ZCTA file** so every zip code has coordinates. It is tab delimited with seven
   columns, and the coordinates are the last two - read the header before you write the `COPY`.
2. **Find the nearest station to each supplier zip code.** Snowflake has a built-in `HAVERSINE`
   function, so there is no need to write the trigonometry by hand. Filter the station index to a
   rough bounding box first: comparing eight zip codes against every weather station on earth is
   a lot of arithmetic to throw away.
3. **Build `supplier_zip_code_weather`** - zip code, date, daily high - and join it to the orders.

One trap in the supplier data: at least one `postalpostalcode` is stored with four characters
where the ZCTA file has five. Pad it before you join or that supplier silently disappears.


In [11]:
# insert code: load the ZCTA zip code / lat / long file
cs.execute(r"""
CREATE OR REPLACE FILE FORMAT STAGE.tsv_zcta
  TYPE = CSV
  FIELD_DELIMITER = '\t'
  SKIP_HEADER = 1
  TRIM_SPACE = TRUE
  EMPTY_FIELD_AS_NULL = TRUE
""")

cs.execute("CREATE OR REPLACE STAGE STAGE.zcta_stage")
zcta_path = os.path.abspath("data/2021_Gaz_zcta_national.txt")
cs.execute(
    f"PUT 'file://{zcta_path}' @STAGE.zcta_stage AUTO_COMPRESS=TRUE OVERWRITE=TRUE"
)

cs.execute("""
CREATE OR REPLACE TABLE CORE.ZCTA (
    geoid        VARCHAR,
    aland        NUMBER,
    awater       NUMBER,
    aland_sqmi   NUMBER(18,3),
    awater_sqmi  NUMBER(18,3),
    intptlat     FLOAT,
    intptlong    FLOAT
)
""")

cs.execute("""
COPY INTO CORE.ZCTA (geoid, aland, awater, aland_sqmi, awater_sqmi, intptlat, intptlong)
FROM (
  SELECT $1::VARCHAR,
         $2::NUMBER,
         $3::NUMBER,
         $4::NUMBER(18,3),
         $5::NUMBER(18,3),
         TRIM($6)::FLOAT,
         TRIM($7)::FLOAT
  FROM @STAGE.zcta_stage
)
FILE_FORMAT = (FORMAT_NAME = STAGE.tsv_zcta)
ON_ERROR = 'ABORT_STATEMENT'
""")

cs.execute("SELECT COUNT(*) FROM CORE.ZCTA")
print("zcta rows (expect 33791):", cs.fetchone())


zcta rows (expect 33791): (33791,)


In [12]:
# insert code: pick the single nearest weather station for each distinct supplier zip code
WEATHER = "SNOWFLAKE_PUBLIC_DATA_FREE.PUBLIC_DATA_FREE"

cs.execute(f"""
CREATE OR REPLACE TABLE CORE.ZIP_STATION AS
WITH zips AS (
    SELECT DISTINCT LPAD(deliverypostalcode, 5, '0') AS zip
    FROM CORE.SUPPLIER
),
pts AS (
    SELECT z.zip, g.intptlat AS lat, g.intptlong AS lon
    FROM zips z
    JOIN CORE.ZCTA g ON g.geoid = z.zip
),
tmax_stations AS (
    SELECT DISTINCT noaa_weather_station_id
    FROM {WEATHER}.NOAA_WEATHER_METRICS_TIMESERIES
    WHERE variable = 'maximum_temperature'
      AND date BETWEEN (SELECT MIN(transactiondate) FROM CORE.PURCHASE_ORDERS_AND_INVOICES)
                   AND (SELECT MAX(transactiondate) FROM CORE.PURCHASE_ORDERS_AND_INVOICES)
),
candidates AS (
    SELECT p.zip,
           s.noaa_weather_station_id,
           ST_DISTANCE(ST_MAKEPOINT(p.lon, p.lat),
                       ST_MAKEPOINT(s.longitude, s.latitude)) AS meters
    FROM pts p
    JOIN {WEATHER}.NOAA_WEATHER_STATION_INDEX s
      ON s.latitude  BETWEEN p.lat - 1 AND p.lat + 1
     AND s.longitude BETWEEN p.lon - 1 AND p.lon + 1
    JOIN tmax_stations t
      ON t.noaa_weather_station_id = s.noaa_weather_station_id
)
SELECT zip, noaa_weather_station_id, meters
FROM candidates
QUALIFY ROW_NUMBER() OVER (PARTITION BY zip ORDER BY meters) = 1
""")

cs.execute("""
SELECT zip, noaa_weather_station_id, ROUND(meters/1000, 1) AS km
FROM CORE.ZIP_STATION ORDER BY zip
""")
for row in cs.fetchall():
    print(row)

('06331', 'USC00063420', 9.0)
('22202', 'USW00013743', 1.8)
('29625', 'USC00387687', 7.1)
('34269', 'USC00080228', 16.6)
('42437', 'USC00128967', 17.6)
('60523', 'USC00115097', 10.3)
('80125', 'USC00054452', 2.8)
('95642', 'USC00048713', 4.9)


In [13]:
# insert code: build supplier_zip_code_weather, then join it to the orders and suppliers
WEATHER = "SNOWFLAKE_PUBLIC_DATA_FREE.PUBLIC_DATA_FREE"

# daily high per distinct supplier zip code
cs.execute(f"""
CREATE OR REPLACE TABLE CORE.SUPPLIER_ZIP_CODE_WEATHER AS
SELECT zs.zip    AS zip_code,
       m.date    AS weather_date,
       m.value   AS max_temp_c
FROM CORE.ZIP_STATION zs
JOIN {WEATHER}.NOAA_WEATHER_METRICS_TIMESERIES m
  ON m.noaa_weather_station_id = zs.noaa_weather_station_id
WHERE m.variable = 'maximum_temperature'
  AND m.date BETWEEN (SELECT MIN(transactiondate) FROM CORE.PURCHASE_ORDERS_AND_INVOICES)
                 AND (SELECT MAX(transactiondate) FROM CORE.PURCHASE_ORDERS_AND_INVOICES)
""")

cs.execute("""
SELECT COUNT(*), COUNT(DISTINCT zip_code || weather_date), COUNT(DISTINCT zip_code)
FROM CORE.SUPPLIER_ZIP_CODE_WEATHER
""")
print("rows / distinct zip+date / zips:", cs.fetchone())

# orders + invoices + supplier + weather, matched on zip and transaction date
cs.execute("""
CREATE OR REPLACE TABLE CORE.ORDERS_INVOICES_WEATHER AS
SELECT poi.purchaseorderid,
       poi.supplierid,
       s.suppliername,
       poi.transactiondate,
       poi.poamount,
       poi.amountexcludingtax,
       poi.invoiced_vs_quoted,
       w.zip_code,
       w.max_temp_c
FROM CORE.PURCHASE_ORDERS_AND_INVOICES poi
JOIN CORE.SUPPLIER s
  ON s.supplierid = poi.supplierid
JOIN CORE.SUPPLIER_ZIP_CODE_WEATHER w
  ON w.zip_code    = LPAD(s.deliverypostalcode, 5, '0')
 AND w.weather_date = poi.transactiondate
""")

cs.execute("SELECT COUNT(*) FROM CORE.ORDERS_INVOICES_WEATHER")
print("matched transactions (of 2072):", cs.fetchone())

rows / distinct zip+date / zips: (8986, 8986, 8)
matched transactions (of 2072): (1497,)


## Part 2 - Questions

Each question has a single numeric answer. Run the query; the query and the number are what is
being marked.

Under each one there is a markdown cell for your own reading of the result. Filling it in is
optional - but try it where you see something worth saying.


### Question 1. Across every purchase order in the data, what is the total value of the goods that were **actually received**? Two decimals.


In [14]:
# insert code: answer the question above with a single query
cs.execute("""
SELECT ROUND(SUM(poamount), 2)
FROM CORE.PURCHASE_ORDER_TOTALS
""")

print(cs.fetchone()[0])

941473291.90


Across the period, the company received goods valued at approximately 941.47 million. This establishes the scale of the purchasing activity represented in the dataset.


### Question 2. Not every row in the supplier transaction XML is an invoice. How many are **not** invoices against a purchase order?


In [15]:
# insert code: answer the question above with a single query
cs.execute("""
SELECT COUNT(*)
FROM CORE.SUPPLIER_TRANSACTIONS
WHERE transactiontypeid <> 5
   OR purchaseorderid IS NULL
""")

print(cs.fetchone()[0])

366


The XML contains 366 transactions that are not purchase-order invoices. These are payment records, demonstrating why the data must be filtered before invoices are compared with purchase orders.


### Question 3. Across all purchase orders that were invoiced, what is the **total amount billed in excess** of the value of goods received?


In [16]:
# insert code: answer the question above with a single query
cs.execute("""
SELECT ROUND(SUM(invoiced_vs_quoted), 2)
FROM CORE.PURCHASE_ORDERS_AND_INVOICES
""")

print(cs.fetchone()[0])

2869980.00


Suppliers billed 2.87 million more than the recorded value of goods received. This difference is concentrated in exactly five purchase orders, while the other 2,067 invoiced orders have no difference, making those five orders clear candidates for investigation.


### Question 4. What share of the total received value comes from the **single largest supplier**? As a percentage, two decimals.


In [17]:
# insert code: answer the question above with a single query
cs.execute("""
WITH supplier_totals AS (
    SELECT supplierid,
           SUM(poamount) AS received_value
    FROM CORE.PURCHASE_ORDER_TOTALS
    GROUP BY supplierid
)
SELECT ROUND(
    100 * MAX(received_value) / SUM(received_value),
    2
)
FROM supplier_totals
""")

print(cs.fetchone()[0])

71.72


The largest supplier accounts for 71.72% of all received value. Because only seven of the 13 suppliers ever appear on an order, this represents substantial concentration among the active suppliers.


### Question 5. Looking at the monthly total value of goods received, which month saw the **largest increase over the month before it**? Answer as `YYYYMM`.

This one needs a window function: the comparison is between a row and the row before it in time.


In [18]:
# insert code: answer the question above with a single query
cs.execute("""
WITH monthly_totals AS (
    SELECT DATE_TRUNC('month', orderdate) AS order_month,
           SUM(poamount) AS received_value
    FROM CORE.PURCHASE_ORDER_TOTALS
    WHERE orderdate IS NOT NULL
    GROUP BY 1
),
monthly_changes AS (
    SELECT order_month,
           received_value
             - LAG(received_value) OVER (ORDER BY order_month) AS increase
    FROM monthly_totals
)
SELECT TO_CHAR(order_month, 'YYYYMM')
FROM monthly_changes
WHERE increase IS NOT NULL
ORDER BY increase DESC
LIMIT 1
""")

print(cs.fetchone()[0])

202203


March 2022 experienced the largest month-over-month increase, rising by 7.73 million from February. The data identifies when the increase occurred, although additional information would be needed to explain its cause.


### Question 6. Between the first and last order date, how many **calendar days** went by with no purchase order at all?

You cannot count rows that are not there - the calendar has to be generated first, then the orders anti-joined against it.


In [19]:
# insert code: answer the question above with a single query
cs.execute("""
WITH bounds AS (
    SELECT MIN(orderdate) AS first_date,
           MAX(orderdate) AS last_date
    FROM CORE.PURCHASE_ORDER_TOTALS
),
offsets AS (
    SELECT ROW_NUMBER() OVER (ORDER BY SEQ4()) - 1 AS day_offset
    FROM TABLE(GENERATOR(ROWCOUNT => 2000))
),
calendar AS (
    SELECT DATEADD(day, o.day_offset, b.first_date) AS calendar_date
    FROM bounds b
    CROSS JOIN offsets o
    WHERE DATEADD(day, o.day_offset, b.first_date) <= b.last_date
),
order_dates AS (
    SELECT DISTINCT orderdate
    FROM CORE.PURCHASE_ORDER_TOTALS
    WHERE orderdate IS NOT NULL
)
SELECT COUNT(*)
FROM calendar c
LEFT JOIN order_dates o
    ON o.orderdate = c.calendar_date
WHERE o.orderdate IS NULL
""")

print(cs.fetchone()[0])

184


Of the 184 days without a purchase order, 104 were Mondays and 77 were Sundays; only three were Saturdays. With 181 of the 184 days falling on Sunday or Monday, the gaps appear to follow a weekly operating pattern rather than occurring randomly.


### Question 7. Of the supplier zip codes in `supplier_case`, which one is **furthest north**?


In [20]:
# insert code: answer the question above with a single query
cs.execute("""
SELECT LPAD(s.deliverypostalcode, 5, '0') AS zip_code
FROM CORE.SUPPLIER s
JOIN CORE.ZCTA z
    ON z.geoid = LPAD(s.deliverypostalcode, 5, '0')
ORDER BY z.intptlat DESC
LIMIT 1
""")

print(cs.fetchone()[0])

60523


ZIP code 60523 is the northernmost supplier ZIP code. This result required padding the four-digit value 6331 to 06331; otherwise, that supplier would have silently failed to join to the ZIP-code gazetteer and been excluded from the comparison.


### Question 8. How many suppliers in `supplier_case` **never appear on a purchase order**?


In [21]:
# insert code: answer the question above with a single query
cs.execute("""
SELECT COUNT(*)
FROM CORE.SUPPLIER s
WHERE NOT EXISTS (
    SELECT 1
    FROM CORE.PURCHASES p
    WHERE p.supplierid = s.supplierid
)
""")

print(cs.fetchone()[0])

6


Six of the 13 suppliers never appear on a purchase order, meaning almost half of the supplier master list was unused during the period. These may be inactive suppliers or vendors retained for purposes other than purchasing.


## Close the connection


In [22]:
# insert code: close the cursor and the connection
cs.close()
conn.close()